# Steam Games — Exploratory Data Analysis

This notebook explores a Steam games dataset covering pricing, genres, review scores, playtime, and release trends.

**Dataset note:** `data/steam_games.csv` here is a **synthetic dataset** generated by `data/generate_data.py`, built to mimic realistic Steam catalog patterns (price by genre, indie vs. non-indie, review-count long tails, a few missing/duplicate rows to clean, etc.). To analyze *real* Steam data instead, download a dataset such as Kaggle's "Steam Store Games" or "Steam Games Dataset", save it as `data/steam_games.csv` with matching column names (or adjust the column names in Section 1), and re-run the notebook.

**Contents**
1. Setup & Data Loading
2. Data Overview & Cleaning
3. Univariate Analysis (price, genre, ratings, playtime)
4. Bivariate & Multivariate Analysis
5. Time Trends
6. Top Games / Publishers
7. Key Takeaways

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 50)

In [ ]:
df = pd.read_csv("data/steam_games.csv", parse_dates=["release_date"])
print(df.shape)
df.head()

## 2. Data Overview & Cleaning
Check shape, types, missing values, and duplicates before doing any analysis.

In [ ]:
df.info()

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
dupes = df.duplicated(subset=["appid"]).sum()
print(f"Duplicate appid rows: {dupes}")

**Cleaning steps taken:**
- Drop duplicate `appid` rows (keep first).
- Fill missing `developer` with `"Unknown"` rather than dropping rows.
- Fill missing `price_usd` with the median price (a handful of rows only).
- Derive `release_year`, `total_ratings`, and `rating_pct` (% positive reviews) for later use.

In [ ]:
df = df.drop_duplicates(subset=["appid"]).copy()
df["developer"] = df["developer"].fillna("Unknown")
df["price_usd"] = df["price_usd"].fillna(df["price_usd"].median())

df["release_year"] = df["release_date"].dt.year
df["total_ratings"] = df["positive_ratings"] + df["negative_ratings"]
df["rating_pct"] = np.where(df["total_ratings"] > 0,
                              df["positive_ratings"] / df["total_ratings"] * 100,
                              np.nan)

print(df.shape)
df.describe(include="number").T

## 3. Univariate Analysis

### 3.1 Price distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df["price_usd"], bins=40, ax=axes[0])
axes[0].set_title("Distribution of Game Prices")
axes[0].set_xlabel("Price (USD)")

sns.histplot(df.loc[df["price_usd"] > 0, "price_usd"], bins=40, ax=axes[1], color="orange")
axes[1].set_title("Price Distribution (Paid Games Only)")
axes[1].set_xlabel("Price (USD)")

plt.tight_layout()
plt.show()

print(f"Free-to-play share: {(df['price_usd'] == 0).mean():.1%}")
print(df["price_usd"].describe())

### 3.2 Genre distribution

In [ ]:
genre_counts = df["primary_genre"].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=genre_counts.values, y=genre_counts.index, hue=genre_counts.index,
            palette="viridis", legend=False, ax=ax)
ax.set_xlabel("Number of Games")
ax.set_title("Games per Primary Genre")
plt.tight_layout()
plt.show()

### 3.3 Review score distribution

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df["rating_pct"].dropna(), bins=30, kde=True, ax=ax)
ax.set_title("Distribution of % Positive Reviews")
ax.set_xlabel("% Positive Reviews")
plt.show()

print(df["rating_pct"].describe())

### 3.4 Playtime distribution

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df["average_playtime_min"] / 60, bins=40, ax=ax, color="seagreen")
ax.set_title("Distribution of Average Playtime")
ax.set_xlabel("Average Playtime (hours)")
plt.show()

## 4. Bivariate & Multivariate Analysis

### 4.1 Price by genre

In [ ]:
order = df.groupby("primary_genre")["price_usd"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="primary_genre", y="price_usd", order=order, ax=ax)
ax.set_title("Price Distribution by Genre")
ax.set_xlabel("Genre")
ax.set_ylabel("Price (USD)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

### 4.2 Indie vs. non-indie: price and reception

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df, x="is_indie", y="price_usd", ax=axes[0])
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Non-Indie", "Indie"])
axes[0].set_title("Price: Indie vs Non-Indie")

sns.boxplot(data=df, x="is_indie", y="rating_pct", ax=axes[1])
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Non-Indie", "Indie"])
axes[1].set_title("Review Score: Indie vs Non-Indie")

plt.tight_layout()
plt.show()

### 4.3 Does price correlate with review score or playtime?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df, x="price_usd", y="rating_pct", hue="is_indie", alpha=0.5, ax=ax)
ax.set_title("Price vs. Review Score")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("% Positive Reviews")
plt.show()

In [ ]:
numeric_cols = ["price_usd", "total_ratings", "rating_pct",
                "average_playtime_min", "median_playtime_min"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Between Numeric Features")
plt.tight_layout()
plt.show()

## 5. Time Trends

In [ ]:
yearly = df.groupby("release_year").agg(
    n_games=("appid", "count"),
    avg_price=("price_usd", "mean"),
    avg_rating=("rating_pct", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.lineplot(data=yearly, x="release_year", y="n_games", marker="o", ax=axes[0])
axes[0].set_title("Games Released per Year")

sns.lineplot(data=yearly, x="release_year", y="avg_price", marker="o", color="darkorange", ax=axes[1])
axes[1].set_title("Average Price by Release Year")

sns.lineplot(data=yearly, x="release_year", y="avg_rating", marker="o", color="seagreen", ax=axes[2])
axes[2].set_title("Average Review Score by Release Year")

plt.tight_layout()
plt.show()

### Genre popularity over time

In [ ]:
top_genres = df["primary_genre"].value_counts().head(5).index
genre_year = (df[df["primary_genre"].isin(top_genres)]
              .groupby(["release_year", "primary_genre"])["appid"].count()
              .reset_index(name="n_games"))

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=genre_year, x="release_year", y="n_games", hue="primary_genre", marker="o", ax=ax)
ax.set_title("Top 5 Genres: Releases per Year")
plt.tight_layout()
plt.show()

## 6. Top Games / Publishers

### 6.1 Most-reviewed games (proxy for popularity)

In [ ]:
top_reviewed = df.sort_values("total_ratings", ascending=False).head(10)
top_reviewed[["name", "primary_genre", "price_usd", "total_ratings", "rating_pct"]]

### 6.2 Highest-rated games with a meaningful review count (>= 100 reviews)

In [ ]:
qualified = df[df["total_ratings"] >= 100]
best_rated = qualified.sort_values("rating_pct", ascending=False).head(10)
best_rated[["name", "primary_genre", "price_usd", "total_ratings", "rating_pct"]]

### 6.3 Publishers by average review score (min. 10 games)

In [ ]:
pub_stats = (df.groupby("publisher")
             .agg(n_games=("appid", "count"), avg_rating=("rating_pct", "mean"),
                  avg_price=("price_usd", "mean"))
             .query("n_games >= 10")
             .sort_values("avg_rating", ascending=False))
pub_stats

## 7. Key Takeaways

- Fill in observations here once you've looked at the charts above, e.g.:
  - Which genres dominate the catalog, and which are priced highest?
  - Do indie games get rated better or worse than non-indie games?
  - Is there any relationship between price and review score?
  - How has the yearly release volume and average price shifted over time?
  - Which publishers consistently produce well-reviewed games?

_This section is intentionally left as a template — replace the bullets above with your own
 findings once you've explored the (or your own real) dataset._